## Databricks Streaming Pipeline
Aiven Kafka -> PySpark Structured Streaming -> Neo4j AuraDB -> Snowflake

| Step | Description |
|------|-------------|
| 0 | Install dependencies |
| 1 | Load credentials and configuration |
| 2 | Verify ca.pem certificate |
| 3 | Neo4j connect and define helpers |
| 4 | Snowflake connect and verify schema |
| 5 | Kafka define streaming reader |
| 6 | Define JSON schema and parse stream |
| 7 | Define `process_batch` enrichment function |
| 8 | Start live streaming pipeline |
| 9 | Monitor and stop stream |

Run each step cell and its debug cell before moving to the next.

### Step 0 - Install Python Dependencies

In [0]:
# Step 0 - Install neo4j Python driver
%pip install neo4j

#### Debug 0 - Confirm neo4j installed

In [0]:
# Debug 0 - Confirm neo4j installed correctly
# Expected: version number printed, no ImportError
import neo4j
from neo4j import GraphDatabase

print(f'neo4j version: {neo4j.__version__}')
print('GraphDatabase import OK')

### Step 1 - Credentials and Configuration

In [0]:
# Step 1 - Load credentials and configuration
import os

def get_env(name, default=None, required=False):
    value = os.getenv(name)
    if value not in (None, ''):
        return value
    if default is not None:
        return default
    if required:
        raise RuntimeError(f'Missing required environment variable: {name}')
    return ''

def get_secret_or_env(env_name, secret_scope, secret_key, default=None, required=False):
    value = os.getenv(env_name)
    if value not in (None, ''):
        return value
    dbutils_obj = globals().get('dbutils')
    if dbutils_obj is not None:
        try:
            return dbutils_obj.secrets.get(scope=secret_scope, key=secret_key)
        except Exception:
            pass
    if default is not None:
        return default
    if required:
        raise RuntimeError(
            f'Missing required secret. Set {env_name} or add {secret_key} to the Databricks secret scope.'
        )
    return ''

# Aiven Kafka
KAFKA_BOOTSTRAP = get_env('KAFKA_BOOTSTRAP_SERVERS', 'datadosekafka-901-datadosedepiproject001.l.aivencloud.com:15816')
KAFKA_TOPIC = get_env('KAFKA_TOPIC', 'DataDose.in')
KAFKA_USERNAME = get_secret_or_env('KAFKA_USERNAME', 'datadose', 'kafka-username', required=True)
KAFKA_PASSWORD = get_secret_or_env('KAFKA_PASSWORD', 'datadose', 'kafka-password', required=True)
KAFKA_CA_PEM_PATH = get_env('KAFKA_CA_PEM_PATH', required=True)

# Neo4j AuraDB
NEO4J_URI = get_env('NEO4J_URI', required=True)
NEO4J_USER = get_secret_or_env('NEO4J_USER', 'datadose', 'neo4j-user', required=True)
NEO4J_PASSWORD = get_secret_or_env('NEO4J_PASSWORD', 'datadose', 'neo4j-password', required=True)

# Snowflake
SNOWFLAKE_SOURCE = 'net.snowflake.spark.snowflake'
sf_options = {
    'sfURL'       : get_env('SNOWFLAKE_URL', required=True),
    'sfUser'      : get_secret_or_env('SNOWFLAKE_USER', 'datadose', 'snowflake-user', required=True),
    'sfPassword'  : get_secret_or_env('SNOWFLAKE_PASSWORD', 'datadose', 'snowflake-password', required=True),
    'sfDatabase'  : get_env('SNOWFLAKE_DATABASE', 'PHARMA_ANALYTICS_DB'),
    'sfSchema'    : get_env('SNOWFLAKE_SCHEMA', 'STAGING'),
    'sfWarehouse' : get_env('SNOWFLAKE_WAREHOUSE', 'PHARMA_WH'),
    'sfRole'      : get_env('SNOWFLAKE_ROLE', 'PYSPARK_ROLE'),
}

# Kafka JAAS config
# The kafkashaded prefix is required on the Spark JVM classpath.
KAFKA_JAAS = (
    f'kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required '
    f'username="{KAFKA_USERNAME}" password="{KAFKA_PASSWORD}";'
)

print('All config variables set.')

#### Debug 1 - Print config (passwords masked)

In [0]:
# Debug 1 - Print config with passwords masked
def mask(s: str) -> str:
    return s[:4] + '****' + s[-4:] if len(s) > 8 else '****'

print('Kafka')
print(f'  Bootstrap   : {KAFKA_BOOTSTRAP}')
print(f'  Topic       : {KAFKA_TOPIC}')
print(f'  Username    : {KAFKA_USERNAME}')
print(f'  Password    : {mask(KAFKA_PASSWORD)}')
print(f'  ca.pem path : {KAFKA_CA_PEM_PATH}')
print()
print('Neo4j')
print(f'  URI      : {NEO4J_URI}')
print(f'  User     : {NEO4J_USER}')
print(f'  Password : {mask(NEO4J_PASSWORD)}')
print()
print('Snowflake')
for k, v in sf_options.items():
    display_v = mask(str(v)) if 'Password' in k or 'password' in k else v
    print(f'  {k:<15}: {display_v}')
print()
print('JAAS (first 60 chars)')
print(f'  {KAFKA_JAAS[:60]}...')

### Step 2 - Verify ca.pem Certificate

In [0]:
# Step 2 - Verify ca.pem exists and copy to /tmp for Spark JVM
import os, shutil

ca_exists = os.path.exists(KAFKA_CA_PEM_PATH)
print(f'ca.pem in Workspace: {KAFKA_CA_PEM_PATH} -> {ca_exists}')

if ca_exists:
    shutil.copy(KAFKA_CA_PEM_PATH, '/tmp/ca.pem')
    KAFKA_CA_PEM_SPARK = '/tmp/ca.pem'
    print('ca.pem copied to /tmp/ca.pem for Spark JVM access')
else:
    print('ca.pem not found')
    print('HOW TO FIX:')
    print('1. Download ca.pem from Aiven Dashboard -> Your Kafka -> Overview')
    print('2. Upload to Databricks Workspace -> your user folder')
    print('3. Update KAFKA_CA_PEM_PATH in Step 1 and re-run')
    raise FileNotFoundError(f'ca.pem not found at {KAFKA_CA_PEM_PATH}')

#### Debug 2 - Inspect ca.pem content

In [0]:
# Debug 1 - Print config with passwords masked
def mask(s: str) -> str:
    if not s:
        return '****'
    return s[:4] + '****' + s[-4:] if len(s) > 8 else '****'

print('Kafka')
print(f'  Bootstrap   : {KAFKA_BOOTSTRAP}')
print(f'  Topic       : {KAFKA_TOPIC}')
print(f'  Username    : {mask(KAFKA_USERNAME)}')
print(f'  Password    : {mask(KAFKA_PASSWORD)}')
print(f'  ca.pem path : {KAFKA_CA_PEM_PATH}')
print()
print('Neo4j')
print(f'  URI      : {NEO4J_URI}')
print(f'  User     : {mask(NEO4J_USER)}')
print(f'  Password : {mask(NEO4J_PASSWORD)}')
print()
print('Snowflake')
for k, v in sf_options.items():
    display_v = mask(str(v)) if k in {'sfUser', 'sfPassword'} else v
    print(f'  {k:<15}: {display_v}')
print()
print('JAAS config is prepared (value hidden)')

### Step 3 - Neo4j Connection and Helper Functions

In [0]:
# Step 3 - Define Neo4j driver and interaction helpers
import logging
from neo4j import GraphDatabase
from typing import List, Dict

logging.getLogger('neo4j.notifications').setLevel(logging.ERROR)

_neo4j_driver = None

def get_neo4j_driver():
    global _neo4j_driver
    if _neo4j_driver is None:
        _neo4j_driver = GraphDatabase.driver(
            NEO4J_URI,
            auth=(NEO4J_USER, NEO4J_PASSWORD),
            max_connection_pool_size=10,
        )
    return _neo4j_driver


def _empty_interaction() -> Dict[str, str]:
    return {
        'interaction_found'        : 'FALSE',
        'interaction_count'        : '0',
        'interacting_drugs'        : '',
        'interaction_severity'     : '',
        'interaction_type'         : '',
        'shared_ingredient'        : '',
        'ingredient_overlap_count' : '0',
    }


def _worst_severity(severities: List[str]) -> str:
    order = {'Major': 1, 'Moderate': 2, 'Minor': 3}
    ranked = sorted([s for s in severities if s in order], key=lambda x: order[x])
    return ranked[0] if ranked else (severities[0] if severities else '')


def check_interactions(new_drug: str, current_drugs: List[str]) -> Dict[str, str]:
    '''Query Neo4j for drug-drug interactions. Returns enrichment fields dict.'''
    if not current_drugs:
        return _empty_interaction()

    driver = get_neo4j_driver()

    interaction_query = '''
        MATCH (d1:Drug)-[r:INTERACTS_WITH]-(d2:Drug)
        WHERE toLower(d1.name) = toLower($new_drug)
          AND ANY(med IN $current_drugs WHERE toLower(d2.name) = toLower(med))
        RETURN d1.name AS drug_a, d2.name AS drug_b,
               r.severity AS severity, r.type AS interaction_type
        ORDER BY CASE r.severity
            WHEN 'Major'    THEN 1
            WHEN 'Moderate' THEN 2
            WHEN 'Minor'    THEN 3
            ELSE 4 END
    '''
    ingredient_query = '''
        MATCH (d1:Drug)-[:HAS_INGREDIENT]->(i:Ingredient)<-[:HAS_INGREDIENT]-(d2:Drug)
        WHERE toLower(d1.name) = toLower($new_drug)
          AND ANY(med IN $current_drugs WHERE toLower(d2.name) = toLower(med))
        RETURN DISTINCT i.name AS ingredient
    '''

    with driver.session() as session:
        interactions = session.run(
            interaction_query, new_drug=new_drug, current_drugs=current_drugs
        ).data()
        shared_ingreds = [
            r['ingredient']
            for r in session.run(
                ingredient_query, new_drug=new_drug, current_drugs=current_drugs
            ).data()
        ]

    if not interactions:
        return {
            **_empty_interaction(),
            'shared_ingredient'        : '|'.join(shared_ingreds),
            'ingredient_overlap_count' : str(len(shared_ingreds)),
        }

    pairs      = [f"{r['drug_a']}\u2194{r['drug_b']}" for r in interactions]
    severities = [r['severity'] for r in interactions if r.get('severity')]
    types      = list({r['interaction_type'] for r in interactions if r.get('interaction_type')})

    return {
        'interaction_found'        : 'TRUE',
        'interaction_count'        : str(len(interactions)),
        'interacting_drugs'        : '|'.join(pairs),
        'interaction_severity'     : _worst_severity(severities),
        'interaction_type'         : '|'.join(types),
        'shared_ingredient'        : '|'.join(shared_ingreds),
        'ingredient_overlap_count' : str(len(shared_ingreds)),
    }


print('Neo4j helper functions defined.')

#### Debug 3a - Neo4j connection ping

In [0]:
# Debug 3a - Neo4j connection ping
# Expected: Neo4j connected and server datetime
try:
    driver = get_neo4j_driver()
    with driver.session() as session:
        row = session.run("RETURN 'Neo4j connected' AS msg, datetime() AS ts").single()
    print(row['msg'])
    print(f"Server time: {row['ts']}")
except Exception as e:
    print(f'Neo4j connection failed: {e}')
    print('COMMON CAUSES:')
    print('Wrong URI - must start with neo4j+s://')
    print('Wrong user - should be the instance ID (e.g. 403ff197)')
    print('Firewall - AuraDB requires outbound TCP on port 7687')

#### Debug 3b - Neo4j graph content check

In [0]:
# Debug 3b - Neo4j graph content check
# Expected: Drug node count > 0, INTERACTS_WITH relationships present
try:
    with driver.session() as session:
        node_counts = session.run(
            'MATCH (n) RETURN labels(n)[0] AS label, count(n) AS cnt '
            'ORDER BY cnt DESC LIMIT 10'
        ).data()
        print('Node counts in Neo4j')
        for row in node_counts:
            label_name = row['label'] if row['label'] else 'Unlabeled'
            print(f'   {label_name:<20} : {row["cnt"]:,}')

        relationship_counts = session.run(
            'MATCH ()-[r]->() RETURN type(r) AS rel_type, count(r) AS cnt '
            'ORDER BY cnt DESC LIMIT 5'
        ).data()
        print()
        print('Relationship counts')
        for row in relationship_counts:
            print(f'   {row["rel_type"]:<25} : {row["cnt"]:,}')

        sample_drugs = session.run('MATCH (d:Drug) RETURN d.name AS name LIMIT 5').data()
        print()
        print('Sample Drug nodes')
        for row in sample_drugs:
            print(f'   - {row["name"]}')
except Exception as e:
    print(f'Error: {e}')

#### Debug 3c - Live interaction lookup test

In [0]:
# Debug 3c - Live interaction lookup dry-run
# Expected: interaction_found=TRUE, severity=Major for warfarin+aspirin
TEST_NEW_DRUG = 'warfarin'
TEST_CURRENT_DRUGS = ['aspirin', 'ibuprofen', 'metformin']

print(f"Testing: '{TEST_NEW_DRUG}' vs {TEST_CURRENT_DRUGS}")
print()

result = check_interactions(TEST_NEW_DRUG, TEST_CURRENT_DRUGS)
print('check_interactions() result')
for k, v in result.items():
    print(f'   {k:<30} : {v}')

print()
empty = check_interactions('some_drug', [])
print('Empty current_drugs case')
print(f"   interaction_found : {empty['interaction_found']}   <- should be FALSE")
print(f"   interaction_count : {empty['interaction_count']}       <- should be 0")

### Step 4 - Snowflake Connection and Schema Verification

In [0]:
# Step 4 - Test Snowflake connection
snowflake_identity_df = (
    spark.read
    .format(SNOWFLAKE_SOURCE)
    .options(**sf_options)
    .option('query', '''
        SELECT CURRENT_USER()      AS SF_USER,
               CURRENT_DATABASE()  AS SF_DATABASE,
               CURRENT_WAREHOUSE() AS SF_WAREHOUSE,
               CURRENT_ROLE()      AS SF_ROLE
    ''')
    .load()
)
print('Snowflake connection established.')

#### Debug 4a - Snowflake identity check

In [0]:
# Debug 4a - Show Snowflake identity
print('Snowflake connection identity')
display(snowflake_identity_df)

expected = {
    'SF_USER'      : 'DATADOSE01',
    'SF_DATABASE'  : 'PHARMA_ANALYTICS_DB',
    'SF_WAREHOUSE' : 'PHARMA_WH',
    'SF_ROLE'      : 'PYSPARK_ROLE',
}
row = snowflake_identity_df.collect()[0]
all_ok = True
for col_name, exp_val in expected.items():
    actual = row[col_name]
    ok = 'OK' if str(actual).upper() == exp_val else 'FAIL'
    if ok == 'FAIL':
        all_ok = False
    print(f'  {ok}  {col_name:<15} : got "{actual}" | expected "{exp_val}"')

if all_ok:
    print()
    print('All Snowflake identity checks passed')

#### Debug 4b - STG_TRANSACTION schema check

In [0]:
# Debug 4b - Confirm STG_TRANSACTION table exists and show schema
# Expected: 26 columns matching the DDL
schema_check_df = (
    spark.read
    .format(SNOWFLAKE_SOURCE)
    .options(**sf_options)
    .option('query', '''
        SELECT COLUMN_NAME, DATA_TYPE
        FROM   INFORMATION_SCHEMA.COLUMNS
        WHERE  TABLE_SCHEMA = 'STAGING'
          AND  TABLE_NAME   = 'STG_TRANSACTION'
        ORDER  BY ORDINAL_POSITION
    ''')
    .load()
)

rows = schema_check_df.collect()
if not rows:
    print('STG_TRANSACTION does not exist in STAGING schema')
    print('Run pharma_snowflake_schema.sql DDL in Snowsight first')
else:
    print(f'STG_TRANSACTION found - {len(rows)} columns:')
    for r in rows:
        print(f'   {r["COLUMN_NAME"]:<35} {r["DATA_TYPE"]}')

### Step 5 - Kafka Structured Streaming Reader

In [0]:
# Step 5 - Define Kafka Structured Streaming reader
# kafka.ssl.truststore.location must point to /tmp/ca.pem because the JVM cannot read /Workspace paths.
# kafka.sasl.jaas.config uses the shaded ScramLoginModule class on the Spark classpath.
kafka_stream_df = (
    spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers',       KAFKA_BOOTSTRAP)
    .option('kafka.security.protocol',       'SASL_SSL')
    .option('kafka.sasl.mechanism',          'SCRAM-SHA-256')
    .option('kafka.sasl.jaas.config',        KAFKA_JAAS)
    .option('kafka.ssl.truststore.type',     'PEM')
    .option('kafka.ssl.truststore.location', KAFKA_CA_PEM_SPARK)
    .option('subscribe',                     KAFKA_TOPIC)
    .option('startingOffsets',               'latest')
    .option('failOnDataLoss',                'false')
    .option('kafka.request.timeout.ms',      '30000')
    .option('kafka.session.timeout.ms',      '10000')
    .load()
)
print('Kafka stream reader defined (not started yet).')

#### Debug 5a - Verify Kafka reader schema

In [0]:
# Debug 5a - Verify Kafka reader schema
print('Kafka stream DataFrame schema')
kafka_stream_df.printSchema()

expected_kafka_cols = {'key','value','topic','partition','offset','timestamp','timestampType'}
missing = expected_kafka_cols - set(kafka_stream_df.columns)
if missing:
    print(f'Missing expected columns: {missing}')
else:
    print('All expected Kafka columns present')

#### Debug 5b - Batch connectivity test (5 real messages)

In [0]:
# Debug 5b - Quick Kafka batch read to confirm real messages arrive
# Ensure simulator.py is running on your local machine first.
# Expected: count > 0, JSON values visible
from pyspark.sql import functions as F

print('Reading up to 5 raw messages from Kafka (batch mode)...')
print('Ensure simulator.py is running: python simulator.py --rate 5')
print()

try:
    kafka_batch_df = (
        spark.read
        .format('kafka')
        .option('kafka.bootstrap.servers',       KAFKA_BOOTSTRAP)
        .option('kafka.security.protocol',       'SASL_SSL')
        .option('kafka.sasl.mechanism',          'SCRAM-SHA-256')
        .option('kafka.sasl.jaas.config',        KAFKA_JAAS)
        .option('kafka.ssl.truststore.type',     'PEM')
        .option('kafka.ssl.truststore.location', KAFKA_CA_PEM_SPARK)
        .option('subscribe',                     KAFKA_TOPIC)
        .option('startingOffsets',               'earliest')
        .option('endingOffsets',                 'latest')
        .load()
        .limit(5)
    )

    count = kafka_batch_df.count()
    print(f"Messages found in topic '{KAFKA_TOPIC}': {count}")

    if count > 0:
        kafka_batch_df.select(
            'offset', 'partition', 'timestamp',
            F.col('value').cast('string').alias('json_value')
        ).show(5, truncate=100)
    else:
        print('0 messages - is simulator.py running?')
        print('Run: python simulator.py --rate 5')

except Exception as e:
    print(f'Kafka batch read failed: {e}')
    print('COMMON CAUSES:')
    print('ca.pem not copied to /tmp (re-run Step 2)')
    print('Wrong SASL password')
    print('spark-sql-kafka JAR not installed on cluster')
    print(f"Topic '{KAFKA_TOPIC}' does not exist in Aiven")

### Step 6 - JSON Schema and Stream Parsing

In [0]:
# Step 6 - Define JSON schema that matches simulator.py output and parse stream
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, ArrayType, LongType
 )

prescription_schema = StructType([
    StructField('transaction_id',  LongType(),              True),
    StructField('patient_id',      IntegerType(),           True),
    StructField('pharmacy_id',     StringType(),            True),
    StructField('pharmacy_city',   StringType(),            True),
    StructField('new_drug',        StringType(),            True),
    StructField('new_drug_dose',   StringType(),            True),
    StructField('new_drug_form',   StringType(),            True),
    StructField('current_drugs',   ArrayType(StringType()), True),
    StructField('patient_age',     IntegerType(),           True),
    StructField('patient_gender',  StringType(),            True),
    StructField('timestamp',       StringType(),            True),
])

parsed_df = (
    kafka_stream_df
    .select(
        F.col('value').cast('string').alias('raw_json'),
        F.col('offset'),
        F.col('timestamp').alias('kafka_timestamp'),
)
    .withColumn('data', F.from_json(F.col('raw_json'), prescription_schema))
    .select('raw_json', 'kafka_timestamp', 'data.*')
)

print('Prescription schema and parsed_df defined.')

#### Debug 6a - Verify parsed schema

In [0]:
# Debug 6a - Verify parsed schema shape
# Expected: 13 cols - 11 simulator fields + raw_json + kafka_timestamp
print('parsed_df schema')
parsed_df.printSchema()

expected_fields = {
    'raw_json', 'kafka_timestamp', 'transaction_id', 'patient_id',
    'pharmacy_id', 'pharmacy_city', 'new_drug', 'new_drug_dose',
    'new_drug_form', 'current_drugs', 'patient_age', 'patient_gender', 'timestamp'
}
actual_fields = set(parsed_df.columns)
missing = expected_fields - actual_fields
extra   = actual_fields - expected_fields

if missing: print(f'Missing fields: {missing}')
else:       print('All expected fields present')
if extra:   print(f'Extra fields: {extra}')

#### Debug 6b - Parse real Kafka messages end-to-end

In [0]:
# Debug 6b - Parse real Kafka messages using prescription_schema
# Expected: all 11 fields populated, no nulls in key columns
print('Parsing real Kafka messages with prescription schema...')

try:
    parsed_batch_df = (
        spark.read
        .format('kafka')
        .option('kafka.bootstrap.servers',       KAFKA_BOOTSTRAP)
        .option('kafka.security.protocol',       'SASL_SSL')
        .option('kafka.sasl.mechanism',          'SCRAM-SHA-256')
        .option('kafka.sasl.jaas.config',        KAFKA_JAAS)
        .option('kafka.ssl.truststore.type',     'PEM')
        .option('kafka.ssl.truststore.location', KAFKA_CA_PEM_SPARK)
        .option('subscribe',                     KAFKA_TOPIC)
        .option('startingOffsets',               'earliest')
        .option('endingOffsets',                 'latest')
        .load()
        .limit(3)
        .select(F.col('value').cast('string').alias('raw_json'), 'offset')
        .withColumn('data', F.from_json(F.col('raw_json'), prescription_schema))
        .select('raw_json', 'data.*')
    )

    rows = parsed_batch_df.collect()
    print(f'Parsed {len(rows)} message(s)')
    print()

    for i, row in enumerate(rows):
        print(f'Message {i+1}')
        for field in ['transaction_id','patient_id','pharmacy_id','pharmacy_city',
                      'new_drug','new_drug_dose','new_drug_form','current_drugs',
                      'patient_age','patient_gender','timestamp']:
            print(f'  {field:<18} : {row[field]}')
        null_fields = [f for f in ['transaction_id','new_drug','pharmacy_id']
                       if row[f] is None]
        if null_fields:
            print(f'  Null in key fields: {null_fields}')
        else:
            print('  Key fields all populated')
        print()

except Exception as e:
    print(f'Parse test failed: {e}')

### Step 7 - Define process_batch Enrichment Function

In [0]:
# Step 7 - Define foreachBatch processing function
# Each micro-batch:
 # 1. Collect rows from Kafka
 # 2. Call Neo4j to check drug-drug interactions per row
 # 3. Compute drug risk score and patient risk score
 # 4. Write enriched rows to Snowflake STAGING.STG_TRANSACTION
import uuid
from datetime import datetime, UTC
from pyspark.sql import DataFrame


def process_batch(batch_df: DataFrame, batch_id: int) -> None:
    rows = batch_df.collect()
    if not rows:
        print(f'   Batch {batch_id}: empty, skipping.')
        return

    batch_uuid = f'KAFKA-{batch_id}-{uuid.uuid4().hex[:8].upper()}'
    enriched_rows = []

    for row in rows:
        new_drug = row['new_drug'] or ''
        current_drugs = [d for d in (row['current_drugs'] or []) if d]

        neo4j_result = check_interactions(new_drug, current_drugs)
        current_meds_count = len(current_drugs)
        polypharmacy_flag = 'TRUE' if current_meds_count >= 5 else 'FALSE'
        interaction_count = int(neo4j_result['interaction_count'])
        severity = neo4j_result['interaction_severity']
        overlap_count = int(neo4j_result['ingredient_overlap_count'])

        severity_score = {'Major': 40, 'Moderate': 20, 'Minor': 10}.get(severity, 0)
        drug_risk_score = min(100.0, round(
            severity_score
            + interaction_count * 5
            + overlap_count * 3
            + (5 if polypharmacy_flag == 'TRUE' else 0),
            2
        ))
        # Guard against None patient_age.
        age_val = row['patient_age'] if row['patient_age'] is not None else 40
        age_multiplier = 1.0 + (age_val - 40) * 0.005
        patient_risk_score = min(100.0, round(drug_risk_score * age_multiplier, 2))
        high_risk_patient = 'TRUE' if patient_risk_score >= 60 else 'FALSE'
        interaction_rate = round(interaction_count / max(1, current_meds_count), 4)

        enriched_rows.append({
            'BATCH_ID'                 : batch_uuid,
            'SOURCE_SYSTEM'            : 'AIVEN_KAFKA',
            'TX_ID'                    : str(row['transaction_id']),
            'PHARMACY'                 : row['pharmacy_id']   or '',
            'CITY'                     : row['pharmacy_city'] or '',
            'IS_NEW_PRESCRIPTION'      : 'New',
            'DRUG'                     : new_drug,
            'CURRENT_MEDS'             : '|'.join(current_drugs),
            'INTERACTION_FOUND'        : neo4j_result['interaction_found'],
            'INTERACTION_COUNT'        : neo4j_result['interaction_count'],
            'INTERACTING_DRUGS'        : neo4j_result['interacting_drugs'],
            'INTERACTION_SEVERITY'     : neo4j_result['interaction_severity'],
            'INTERACTION_TYPE'         : neo4j_result['interaction_type'],
            'ACTIVE_INGREDIENT_MATCH'  : 'TRUE' if overlap_count > 0 else 'FALSE',
            'SHARED_INGREDIENT'        : neo4j_result['shared_ingredient'],
            'INGREDIENT_OVERLAP_COUNT' : neo4j_result['ingredient_overlap_count'],
            'CURRENT_MEDS_COUNT'       : str(current_meds_count),
            'POLYPHARMACY_FLAG'        : polypharmacy_flag,
            'HIGH_RISK_PATIENT'        : high_risk_patient,
            'DRUG_RISK_SCORE'          : str(drug_risk_score),
            'PATIENT_RISK_SCORE'       : str(patient_risk_score),
            'INTERACTION_RATE'         : str(interaction_rate),
            'RAW_RECORD'               : row['raw_json'],
        })

    enriched_df = spark.createDataFrame(enriched_rows)
    (
        enriched_df.write
        .format(SNOWFLAKE_SOURCE)
        .options(**sf_options)
        .option('dbtable', 'STAGING.STG_TRANSACTION')
        .mode('append')
        .save()
    )

    interaction_count_total = sum(1 for r in enriched_rows if r['INTERACTION_FOUND'] == 'TRUE')
    high_risk_count = sum(1 for r in enriched_rows if r['HIGH_RISK_PATIENT'] == 'TRUE')
    print(f'  Batch {batch_id} [{batch_uuid}] -> {len(enriched_rows)} rows written')
    print(f'     Interactions detected : {interaction_count_total}')
    print(f'     High-risk patients    : {high_risk_count}')


print('process_batch function defined.')

#### Debug 7 - Dry-run process_batch on 5 real Kafka messages

In [0]:
# Debug 7 - Dry-run process_batch on 5 real Kafka messages
# Reads from Kafka as batch, runs full enrichment, writes to Snowflake
# Rows will have BATCH_ID = 'KAFKA-9999-*' so they are easy to identify
# Expected: rows in Snowflake with interaction/risk fields populated
print('Running dry-run of process_batch on 5 real Kafka messages...')
print('This will write to Snowflake STAGING.STG_TRANSACTION (BATCH_ID=KAFKA-9999-*)')
print()

try:
    dry_run_df = (
        spark.read
        .format('kafka')
        .option('kafka.bootstrap.servers',       KAFKA_BOOTSTRAP)
        .option('kafka.security.protocol',       'SASL_SSL')
        .option('kafka.sasl.mechanism',          'SCRAM-SHA-256')
        .option('kafka.sasl.jaas.config',        KAFKA_JAAS)
        .option('kafka.ssl.truststore.type',     'PEM')
        .option('kafka.ssl.truststore.location', KAFKA_CA_PEM_SPARK)
        .option('subscribe',                     KAFKA_TOPIC)
        .option('startingOffsets',               'earliest')
        .option('endingOffsets',                 'latest')
        .load()
        .limit(5)
        .select(F.col('value').cast('string').alias('raw_json'), 'timestamp')
        .withColumn('data', F.from_json(F.col('raw_json'), prescription_schema))
        .select('raw_json', F.col('timestamp').alias('kafka_timestamp'), 'data.*')
    )

    process_batch(dry_run_df, batch_id=9999)

    print()
    print('Verifying rows written to Snowflake')
    df_written = (
        spark.read
        .format(SNOWFLAKE_SOURCE)
        .options(**sf_options)
        .option('query', '''
            SELECT TX_ID, DRUG, CITY, INTERACTION_FOUND,
                   INTERACTION_SEVERITY, HIGH_RISK_PATIENT,
                   PATIENT_RISK_SCORE, BATCH_ID
            FROM   STAGING.STG_TRANSACTION
            WHERE  BATCH_ID LIKE 'KAFKA-9999-%'
            ORDER  BY LOAD_TIMESTAMP DESC
        ''')
        .load()
    )
    df_written.show(truncate=False)

    count = df_written.count()
    if count > 0:
        print(f'{count} row(s) confirmed in Snowflake')
    else:
        print('0 rows - check Snowflake permissions or Kafka message availability')

except Exception as e:
    print(f'Dry-run failed: {e}')

### Step 8 - Start Live Streaming Pipeline
Only run this after all debug cells above have passed.

In [0]:
# Step 8 - Start the live Kafka -> Neo4j -> Snowflake streaming pipeline
CHECKPOINT_PATH = 'dbfs:/tmp/pharma_pipeline/checkpoints/kafka_to_snowflake'

query = (
    parsed_df.writeStream
    .foreachBatch(process_batch)
    .option('checkpointLocation', CHECKPOINT_PATH)
    .trigger(processingTime='10 seconds')
    .start()
)

print('Streaming pipeline STARTED')
print(f'   Query ID   : {query.id}')
print(f'   Checkpoint : {CHECKPOINT_PATH}')
print(f'   Trigger    : every 10 seconds')
print()
print('Use Debug 8a/8b/8c/8d cells below to monitor the stream')

### Step 9 - Monitor and Stop Stream

#### Debug 8a - Stream status

In [0]:
# Debug 8a - Check stream status (run any time while stream is active)
# Expected: isActive=True, message showing processing or waiting
import time

print(f'Stream active : {query.isActive}')
print(f'Status        : {query.status}')
print()

if query.lastProgress:
    p = query.lastProgress
    print('Last progress')
    print(f"  Batch ID           : {p.get('batchId', '-')}")
    print(f"  Input rows/sec     : {p.get('inputRowsPerSecond', '-')}")
    print(f"  Processed rows/sec : {p.get('processedRowsPerSecond', '-')}")
    sources = p.get('sources', [{}])
    if sources:
        print(f"  Kafka end offset   : {sources[0].get('endOffset', '-')}")
else:
    print('No progress yet - waiting for first batch...')

#### Debug 8b - Watch live batches for 60 seconds

In [0]:
# Debug 8b - Watch live batches for 60 seconds (6 checks x 10s)
# Expected: batch counts increasing, rows being written to Snowflake
print('Watching stream for 60 seconds (6 checks x 10s)...')
print('Ensure simulator.py is running!\n')

for i in range(6):
    time.sleep(10)
    status = query.status
    prog   = query.lastProgress or {}
    batch  = prog.get('batchId', '-')
    rows_s = prog.get('inputRowsPerSecond', '-')
    print(
        f'  [{i+1}/6]  active={query.isActive}  '
        f'batch={batch}  input_rows/s={rows_s}  '
        f"status={status.get('message', '-')}"
    )

#### Debug 8c - Latest rows written to Snowflake

In [0]:
# Debug 8c - Read latest rows written to Snowflake by the live stream
# Expected: recent rows with BATCH_ID not starting with 'KAFKA-9999-'
live_stream_df = (
    spark.read
    .format(SNOWFLAKE_SOURCE)
    .options(**sf_options)
    .option('query', '''
        SELECT TX_ID, DRUG, CITY,
               INTERACTION_FOUND, INTERACTION_SEVERITY,
               HIGH_RISK_PATIENT, PATIENT_RISK_SCORE,
               POLYPHARMACY_FLAG, BATCH_ID, LOAD_TIMESTAMP
        FROM   STAGING.STG_TRANSACTION
        WHERE  BATCH_ID NOT LIKE 'KAFKA-9999-%'
        ORDER  BY LOAD_TIMESTAMP DESC
        LIMIT  20
    ''')
    .load()
)

count = live_stream_df.count()
print(f'{count} live stream row(s) in Snowflake')
live_stream_df.show(truncate=False)

#### Debug 8d - Pipeline health summary

In [0]:
# Debug 8d - Summary stats from Snowflake (final health check)
# Expected: total records, interaction rate, high-risk count, avg risk score
pipeline_stats_df = (
    spark.read
    .format(SNOWFLAKE_SOURCE)
    .options(**sf_options)
    .option('query', '''
        SELECT
            COUNT(*)                                                     AS TOTAL_RECORDS,
            SUM(CASE WHEN INTERACTION_FOUND = 'TRUE' THEN 1 ELSE 0 END) AS INTERACTIONS_DETECTED,
            SUM(CASE WHEN HIGH_RISK_PATIENT = 'TRUE' THEN 1 ELSE 0 END) AS HIGH_RISK_PATIENTS,
            SUM(CASE WHEN POLYPHARMACY_FLAG = 'TRUE' THEN 1 ELSE 0 END) AS POLYPHARMACY_CASES,
            ROUND(AVG(PATIENT_RISK_SCORE::FLOAT), 2)                    AS AVG_RISK_SCORE,
            MAX(LOAD_TIMESTAMP)                                          AS LAST_WRITE
        FROM STAGING.STG_TRANSACTION
    ''')
    .load()
)

print('Pipeline Health Summary')
pipeline_stats_df.show(truncate=False)

#### Stop the stream

In [0]:
# Step 9 - Stop the streaming query
query.stop()
print('Stream stopped.')
print(f'   Final status : {query.status}')